# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, following the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure 'mlcroissant' is installed. Restart kernel after install if necessary.
!pip install --quiet mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"Date published: {metadata.datePublished}")
print(f"License: {metadata.license}")

## 2. Data Overview
List and describe available **record sets**, **fields**, and their `@id`s, referencing all entities by their `@id` from the Croissant schema.

In [ ]:
# List all available record sets (referenced by @id)
record_sets = list(dataset.record_sets.keys())  # These are @id values
print("Available record sets:")
for rs_id in record_sets:
    record_set = dataset.record_sets[rs_id]
    print(f"- @id: {rs_id}, name: {record_set.name}")

# For each record set, list the fields by @id
for rs_id in record_sets:
    record_set = dataset.record_sets[rs_id]
    print(f"\nFields for record set @id '{rs_id}' ({record_set.name}):")
    for field_id, field in record_set.fields.items():
        print(f"  - Field @id: {field_id}, name: {field.name}, dataType: {field.data_type}")

## 3. Data Extraction
Load data from each record set into pandas DataFrames for analysis. All entities are referenced by their `@id` fields.

In [ ]:
# Extract data from each record set using its @id
dataframes = {}
for rs_id in record_sets:
    # Create DataFrame with records for this record set
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records for record set @id={rs_id}, columns:")
    print(df.columns.tolist())
    display(df.head(3))

## 4. Exploratory Data Analysis (EDA)
Perform basic data filtering, normalization, and grouping using the DataFrame. All columns and operations are referenced by `@id`.

We will select the main record set with clinical records for demonstration. Fields may include age, gender, MSI status, anatomical site, etc. Please substitute actual `@id` values (from the earlier overview) as needed.

In [ ]:
# For demonstration, select the first (and likely only) main record set
main_rs_id = record_sets[0]
df = dataframes[main_rs_id]

# Let's identify a likely numeric field by inspecting DataFrame dtypes and choose by @id
numeric_fields = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
print("Numeric fields available:", numeric_fields)

# Use the first numeric field by its @id (update as appropriate based on actual schema)
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    # Filtering: keep records where value > 10 (adjust as meaningful for the field)
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} found")
    display(filtered_df[[numeric_field_id]].head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized '{numeric_field_id}':")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a categorical/grouping field (commonly 'sex', 'MSI status', etc)
    potential_group_fields = df.select_dtypes(include=['object', 'category']).columns.tolist()
    print("Potential group fields:", potential_group_fields)
    # Use first group field
    if potential_group_fields:
        group_field_id = potential_group_fields[0]
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Average {numeric_field_id} grouped by {group_field_id}:")
        print(grouped)
else:
    print("No numeric fields detected for EDA.")

## 5. Visualization
Plot the distribution of a numeric variable and the group means (if applicable).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id], kde=True, bins=20)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    if potential_group_fields:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
- This notebook showed step-by-step how to load, inspect, and analyze a Croissant dataset with `mlcroissant` using all `@id`s as references.
- You can adapt these steps for richer EDA or downstream modeling according to the schema documentation.
- Consult the record set and field overviews to ensure all references are by their `@id`, ensuring reproducibility and schema alignment.